In [ ]:
!pip install pandas matplotlib seaborn wordcloud nltk pymorphy2 scikit-learn -q
!pip install spacy -q
!python -m spacy download ru_core_news_sm -q

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json
import os
from google.colab import files

# Загрузка стоп-слов для русского языка
nltk.download('stopwords', quiet=True)
russian_stopwords = set(stopwords.words('russian'))

print("Первичный анализ текстовых данных (суммаризация)")

# Шаг 1. Загрузка данных
file_path = '/content/gazeta_test.jsonl'

data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Пропущена повреждённая строка: {line[:50]}...")
                continue

df = pd.DataFrame(data)

print("Датасет загружен!")
print(f"Размер датасета: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
print("\nПервые 5 строк:")
print(df.head())

# Шаг 2. Очистка текста
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^а-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df['clean_summary'] = df['summary'].apply(clean_text)

print("\nПример очищенного текста (первые 300 символов):")
print(df['clean_text'].iloc[0][:300])


# Шаг 3. Лемматизация (пример для первых 20 текстов)
nlp = spacy.load("ru_core_news_sm")

def lemmatize_text(text):
    if not isinstance(text, str):
        return ""
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc])

df_sample = df.head(20).copy()
df_sample['lemmatized_text'] = df_sample['clean_text'].apply(lemmatize_text)

print("\nПример лемматизации (первые 100 символов):")
print("Исходный:", df_sample['clean_text'].iloc[0][:100])
print("Лемматизированный:", df_sample['lemmatized_text'].iloc[0][:100])


# Шаг 4. Подсчет частоты слов и облако слов
def plot_wordcloud(text_data, title):
    all_text = ' '.join(text_data)
    words = [word for word in all_text.split() if word not in russian_stopwords]
    filtered_text = ' '.join(words)

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        max_words=100
    ).generate(filtered_text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()
    return plt.gcf()

# Сохраняем фигуры для последующего сохранения
fig_wc_text = plot_wordcloud(df['clean_text'], "Облако слов для статей (без стоп-слов)")
fig_wc_summary = plot_wordcloud(df['clean_summary'], "Облако слов для саммари (без стоп-слов)")


# Шаг 5. Удаление стоп-слов
def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in russian_stopwords]
    return ' '.join(filtered)

df['text_no_stopwords'] = df['clean_text'].apply(remove_stopwords)
df['summary_no_stopwords'] = df['clean_summary'].apply(remove_stopwords)

print("\nПример текста без стоп-слов (первые 200 символов):")
print(df['text_no_stopwords'].iloc[0][:200])


# Шаг 6. Вычисление TF-IDF слов
corpus_for_tfidf = df['clean_text'].head(50).tolist()
stop_words_list = list(russian_stopwords)

tfidf = TfidfVectorizer(max_features=20, stop_words=stop_words_list)
tfidf_matrix = tfidf.fit_transform(corpus_for_tfidf)

feature_names = tfidf.get_feature_names_out()
tfidf_sum = np.array(tfidf_matrix.sum(axis=0)).flatten()

tfidf_df = pd.DataFrame({
    'word': feature_names,
    'tfidf_score': tfidf_sum
}).sort_values('tfidf_score', ascending=False)

print("\nТоп-10 слов по TF-IDF в статьях:")
print(tfidf_df.head(10))

# Визуализация TF-IDF
plt.figure(figsize=(10, 6))
plt.barh(tfidf_df['word'].head(10), tfidf_df['tfidf_score'].head(10))
plt.xlabel('Суммарный TF-IDF')
plt.title('Топ-10 важных слов в статьях (TF-IDF)')
plt.gca().invert_yaxis()
fig_tfidf = plt.gcf()
plt.show()


# Шаг 7. Реализация информационного поиска по корпусу
corpus_size = min(200, len(df))
stop_words_list = list(russian_stopwords)

tfidf_full = TfidfVectorizer(max_features=1000, stop_words=stop_words_list)
tfidf_full_matrix = tfidf_full.fit_transform(df['clean_text'].head(corpus_size))

def search_documents(query, top_n=3):
    query_clean = clean_text(query)
    query_vector = tfidf_full.transform([query_clean])
    similarities = cosine_similarity(query_vector, tfidf_full_matrix).flatten()
    top_indices = similarities.argsort()[-top_n:][::-1]

    print(f"\nПоиск по запросу: '{query}'")
    print("-" * 50)
    for i, idx in enumerate(top_indices):
        score = similarities[idx]
        print(f"{i+1}. Сходство: {score:.4f}")
        print(f"   Статья: {df['text'].iloc[idx][:150]}...")
        print(f"   Резюме: {df['summary'].iloc[idx][:100]}...")
        print()

search_documents("экономика России", top_n=3)
search_documents("спорт и футбол", top_n=3)
search_documents("новые технологии", top_n=3)


# Шаг 8. Сохранение результатов (CSV и изображения)


# Сохранение обработанного датасета
df.to_csv('/content/gazeta_summaries_processed.csv', index=False)

# Сохранение изображений
fig_wc_text.savefig('/content/wordcloud_text.png', dpi=150, bbox_inches='tight')
fig_wc_summary.savefig('/content/wordcloud_summary.png', dpi=150, bbox_inches='tight')
fig_tfidf.savefig('/content/tfidf_top10.png', dpi=150, bbox_inches='tight')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 37.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Первичный анализ текстовых данных (суммаризация)
